# STAC AI Workshop · NYC · 21 October 2026

## 04 · Adaptation lab: data, learning and blind spots

**Participant workbook · Matthew Dixon, PhD, FRM**

### BeliefLens · STAC AI Workshop · NYC · 21 October 2026

## Learning objectives
1. Prepare financial instruction examples
2. Separate training and evaluation data
3. Train and compare a small model
4. Diagnose a missing feature and generalization limits

All companies, numbers, labels and market observations in these notebooks are synthetic teaching examples. Offline outputs come from explicit deterministic rules, not an LLM. The same JavaScript case engine runs in the browser and through this Python client. Only a cell explicitly using `live=True` calls Together AI. No financial action is executed.

**Requirements:** Python 3.10+, Node.js 20+, Jupyter. No model weights or API key are needed for the default exercises. Run cells in order.

## Open and run this notebook
[Open in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/04_adaptation_lab_classifier_reference.ipynb) · [View on GitHub](https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/04_adaptation_lab_classifier_reference.ipynb)

1. Run the **COLAB_BOOTSTRAP** cell first. It downloads public helpers and saved fixtures when using Colab; it does nothing in local JupyterLab.
2. Run the named code cells from top to bottom. Leave live, fresh-inference and GPU switches off initially.
3. References such as **C03:5** mean line 5 inside code cell C03. The guide quotes the exact code so you can find it without editor line numbers. Enable line numbers in your editor preferences if available.
4. Read **Answer checks** below to compare the default result. Edited inputs and live runs can differ; the stated conditions explain what should remain true.

Saved results, constructed examples, rule checks and fresh model calls are identified separately. Do not save API keys in cells or outputs.

In [ ]:
# COLAB_BOOTSTRAP · download the public workshop helpers and saved fixtures.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    import os, subprocess
    workshop_root = Path("/content/STAC_AI_Bootcamp")
    if not (workshop_root / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mfrdixon/STAC_AI_Bootcamp.git", str(workshop_root)], check=True)
    os.chdir(workshop_root / "notebooks")
    sys.path.insert(0, str(workshop_root))
    sys.path.insert(0, str(workshop_root / "notebooks"))
    print("Workshop helpers and saved fixtures ready. No model API call was made.")

<!-- WORKSHOP-CODE-GUIDE C01 -->
### Code cell C01 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C01:4** | `root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_client.py").exists()), None)` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C01 · code cell
from pathlib import Path
import sys, json, math, random, statistics
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_client.py").exists()), None)
if root is None:
    raise RuntimeError("Extract the complete workshop ZIP and open Jupyter inside its folder.")
if str(root) not in sys.path: sys.path.insert(0, str(root))
from lab_client import run, compact, save
print("Shared engine ready:", compact(run()))

## Build examples with explicit targets
These records could inform SFT dataset construction, subject to expert review. Labels come from the declared teaching rule, not from another LLM. The numerical exercise trains a logistic classifier, **not a language model**. It exposes why the features and examples available to a learner matter.

<!-- WORKSHOP-CODE-GUIDE C02 -->
### Code cell C02 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C02:2** | `rng = random.Random(42)` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C02 · code cell
rng = random.Random(42)
rows = []
for i in range(240):
    leverage = rng.uniform(1, 5)
    disruption = int(rng.random() < 0.3)
    label = int(leverage > 3.5 or disruption == 1)
    rows.append({"id":f"case-{i}", "issuer":f"issuer-{i//8}", "leverage":leverage, "disruption":disruption, "review":label})
train = [r for r in rows if int(r["issuer"].split("-")[1]) < 20]
test = [r for r in rows if int(r["issuer"].split("-")[1]) >= 20]
assert not ({r["issuer"] for r in train} & {r["issuer"] for r in test})
sample = train[0]
instruction = {"messages":[{"role":"user","content":f"Assess leverage {sample['leverage']:.2f}x; disruption={sample['disruption']}"},{"role":"assistant","content":"REVIEW" if sample["review"] else "NO EXCEPTION UNDER THIS RULE"}]}
print(json.dumps(instruction, indent=2))
print("Training / held out:", len(train), len(test))

<!-- WORKSHOP-CODE-GUIDE C03 -->
### Code cell C03 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C03:2** | `def features(row, include_disruption):` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C03 · code cell
def features(row, include_disruption):
    return [1., row["leverage"]-3.5] + ([float(row["disruption"])] if include_disruption else [])
def probability(w, x):
    z = max(-30, min(30, sum(a*b for a,b in zip(w,x))))
    return 1/(1+math.exp(-z))
def fit(data, include_disruption):
    w = [0.] * (3 if include_disruption else 2)
    for epoch in range(800):
        gradient = [0.]*len(w)
        for row in data:
            x=features(row,include_disruption); error=probability(w,x)-row["review"]
            for j in range(len(w)): gradient[j]+=error*x[j]/len(data)
        w=[v-0.15*g for v,g in zip(w,gradient)]
    return w
def evaluate(w, data, include_disruption):
    predictions=[int(probability(w,features(r,include_disruption))>=0.5) for r in data]
    misses=sum(p==0 and r["review"]==1 for p,r in zip(predictions,data))
    return {"accuracy":sum(p==r["review"] for p,r in zip(predictions,data))/len(data), "missed_review":misses}
for full in [False,True]:
    w=fit(train,full)
    print("Includes disruption:",full,"train:",evaluate(w,train,full),"test:",evaluate(w,test,full))

## Compare shared-engine configurations
These are controller experiments, separate from the trained classifier. A calculator can improve numerical traceability while the release decision remains blocked by a missing approval.

<!-- WORKSHOP-CODE-GUIDE C04 -->
### Code cell C04 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C04:2** | `for mode in ["model_only","rag","rag_tools"]:` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C04 · code cell
for mode in ["model_only","rag","rag_tools"]:
    print(compact(run({"mode":mode,"disruption":False})))

## Learning check
Which is the main lesson of adding the disruption feature? **A:** more parameters always help; **B:** relevant state omitted from inputs can cause systematic misses; **C:** classification accuracy proves governance safety.

**Answer: B.** The split prevents issuer overlap, but these synthetic data still share a simple generating process. Real evaluation needs realistic temporal, issuer and failure-mode shifts.

**Exercise:** create an additional holdout with changed covenant definitions. Which labels or features would need review?

## What you have demonstrated
- **Prepare financial instruction examples:** Constructed source/target examples with declared labels
- **Separate training and evaluation data:** Prevented issuer overlap
- **Train and compare a small model:** Actually fitted two logistic models and measured held-out errors
- **Diagnose a missing feature and generalization limits:** Compared a model with and without a risk-relevant feature

**Exit ticket:** State one result, the evidence supporting it, and one limit that prevents you from claiming more. Save a run JSON or your numerical results as the experiment record.

## Answer checks · default configuration

- The generated label is 1 when leverage exceeds 3.5 **or** a disruption is present; otherwise 0. These are synthetic rule-generated labels, not independent market outcomes.
- Adding the disruption feature exposes relevant state that leverage-only input cannot represent. It does not prove that more parameters always help. The learning-check answer is the option about **relevant missing information**.
- This cell trains a small illustrative classifier, not a transformer or LoRA adapter. Keep that distinction explicit.
- The shared-engine calls are separate controller demonstrations. A valid calculator result does not establish that all evidence and policy checks passed.
- Use the printed held-out comparison for the seeded run; changing the seed or data changes the exact result.

If your result differs, check which input, selected case, data version or optional switch changed. For a live model, inspect actual tool calls and returned evidence; do not assume the recorded result is guaranteed.